In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List, Tuple, Union

import duckdb
import pandas as pd

true_path = Path("brown_outputs.json")
false_path = Path("negative_outputs.json")

In [2]:
# ---- parser for "quasi-JSON": repeated JSON objects, not wrapped in an array ----
def parse_quasi_json_objects(text: str) -> List[Dict[str, Any]]:
    """
    Parse a string containing multiple JSON objects back-to-back with arbitrary whitespace/newlines.
    Ignores '...' lines and other non-JSON noise.
    """
    # Remove obvious noise lines like "..." that often appear in snippets/logs
    filtered_lines = text.splitlines()
    cleaned = "\n".join([ln for ln in filtered_lines if ln.strip() != "..."])

    decoder = json.JSONDecoder()
    idx = 0
    n = len(cleaned)
    objs: List[Dict[str, Any]] = []

    while idx < n:
        # Skip whitespace
        while idx < n and cleaned[idx].isspace():
            idx += 1
        if idx >= n:
            break

        # If we're not at a JSON object start, skip forward until we are
        if cleaned[idx] != "{":
            idx += 1
            continue

        obj, end = decoder.raw_decode(cleaned, idx)
        if not isinstance(obj, dict):
            raise ValueError(f"Expected JSON object (dict) at pos {idx}, got {type(obj)}")
        objs.append(obj)
        idx = end

    return objs

In [ ]:
con = duckdb.connect(database=":memory:")
a =  """paper_type.screening::BOOLEAN AS screening,
        paper_type.known::BOOLEAN     AS known,
        screens.fragment::BOOLEAN     AS fragment,
        screens.undirected::BOOLEAN   AS undirected,
        screens.directed::BOOLEAN     AS directed,
        screens.virtual::BOOLEAN      AS virtual,
        screens.del::BOOLEAN          AS del,
        title::VARCHAR                AS title,"""
b =  """CASE WHEN paper_type.screening AND label THEN 'TP'
            WHEN paper_type.screening AND NOT label THEN 'FP'
            WHEN NOT paper_type.screening AND NOT label THEN 'TN'
            WHEN NOT paper_type.screening AND label THEN 'FN' END
        AS result
"""
con.execute(f"""
    CREATE OR REPLACE TABLE examples AS
    SELECT 
        {a}
        True                          AS label,
        {b}
    FROM read_json_auto('{str(true_path)}')
    UNION ALL
    SELECT 
        {a}
        False                         AS label,
        {b}
    FROM read_json_auto('{str(false_path)}');
""")

In [4]:
cols = ["screening","known","fragment","undirected","directed","virtual","del","label"]
display(con.execute(
    " UNION ALL\n".join(
    f"""SELECT '{c}' AS column, SUM(CASE WHEN {c} THEN 1 ELSE 0 END) AS true,
    SUM(CASE WHEN NOT {c} THEN 1 ELSE 0 END) AS false FROM examples"""
    for c in cols
)).df())
display(con.execute("DESCRIBE examples").df())
display(con.execute("SELECT * FROM examples").df())

,column,true,false
0,screening,43.0,1.0
1,known,7.0,37.0
2,fragment,0.0,43.0
3,undirected,32.0,11.0
4,directed,0.0,43.0
5,virtual,1.0,42.0
6,del,1.0,42.0
7,label,44.0,0.0


,column_name,column_type,null,key,default,extra
0,screening,BOOLEAN,YES,None,None,None
1,known,BOOLEAN,YES,None,None,None
2,fragment,BOOLEAN,YES,None,None,None
3,undirected,BOOLEAN,YES,None,None,None
4,directed,BOOLEAN,YES,None,None,None
5,virtual,BOOLEAN,YES,None,None,None
6,del,BOOLEAN,YES,None,None,None
7,title,VARCHAR,YES,None,None,None
8,label,BOOLEAN,YES,None,None,None
9,result,VARCHAR,YES,None,None,None


,screening,known,fragment,undirected,directed,virtual,del,title,label,result
0,True,False,False,True,False,False,False,"Discovery of N-(4-(2,4-Difluorophenoxy)-3-(6-m...",True,TP
1,True,False,False,True,False,False,False,Identification of a Benzoisoxazoloazepine Inhi...,True,TP
2,True,True,False,True,False,False,False,Discovery of a Novel and Selective Indoleamine...,True,TP
3,True,False,False,False,False,False,False,"5-(4,6-Dimorpholino-1,3,5-triazin-2-yl)-4-(tri...",True,TP
4,True,True,False,True,False,False,False,The Rational Design of Selective Benzoxazepin ...,True,TP
5,True,False,False,True,False,False,False,"Discovery of 4-((3'R,4'S,5'R)-6″-Chloro-4'-(3-...",True,TP
6,True,False,False,False,False,False,False,"Discovery of N-((3R,4R)-4-Fluoro-1-(6-((3-meth...",True,TP
7,True,False,False,False,False,True,False,Discovery of [5-Amino-1-(2-methyl-3H-benzimida...,True,TP
8,True,False,False,True,False,False,False,Discovery of the Irreversible Covalent FGFR In...,True,TP
9,True,True,False,False,False,False,False,Optimization of Orally Bioavailable Enhancer o...,True,TP


In [5]:
# Compute Accuracy, Precision, Recall, Specificity from examples.result ∈ {TP, FP, TN, FN}

row = con.execute("""
SELECT
  SUM(CASE WHEN result = 'TP' THEN 1 ELSE 0 END) AS TP,
  SUM(CASE WHEN result = 'FP' THEN 1 ELSE 0 END) AS FP,
  SUM(CASE WHEN result = 'TN' THEN 1 ELSE 0 END) AS TN,
  SUM(CASE WHEN result = 'FN' THEN 1 ELSE 0 END) AS FN
  FROM examples
  WHERE screening   IS NOT NULL
    AND label       IS NOT NULL
""").fetchone()

TP, FP, TN, FN = map(int, row)
N = TP + FP + TN + FN

def safe_div(num, den):
    return float('nan') if den == 0 else num / den

accuracy     = safe_div(TP + TN, N)
precision    = safe_div(TP, TP + FP)
recall       = safe_div(TP, TP + FN)
specificity  = safe_div(TN, TN + FP)

print(f"TP={TP} FP={FP} TN={TN} FN={FN} (N={N})")
print(f"Accuracy    : {accuracy:.6f}")
print(f"Precision   : {precision:.6f}")
print(f"Recall      : {recall:.6f}")
print(f"Specificity : {specificity:.6f}")

TP=43 FP=0 TN=0 FN=1 (N=44)
Accuracy    : 0.977273
Precision   : 1.000000
Recall      : 0.977273
Specificity : nan
